# Ordered Logistic Regression Results for Adoption Predictors of Indigenous and Modern Knowledge in Rangeland Management Practices, Northern Kenya Exploration with `mlcroissant`

This notebook provides an end-to-end guide for loading and exploring the FAIRˆ² dataset using the [mlcroissant](https://mlcroissant.readthedocs.io/) library, based on a Croissant schema.

### Dataset Source
The dataset source is provided via a Croissant schema URL:

`https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json`

In [ ]:
# Ensure mlcroissant library is installed
!pip install mlcroissant --quiet

## 1. Data Loading

Load dataset metadata and available record sets using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import pprint

# Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"Dataset loaded: {metadata.name}\n")
print(f"Description: {metadata.description}\n")
print(f"Authors: {getattr(metadata, 'author', None)}\n")
print(f"Published: {getattr(metadata, 'datePublished', None)}\n")
print(f"License: {getattr(metadata, 'license', None)}")

## 2. Data Overview
Review available record sets, their `@id`s, and contained fields.

In [ ]:
# List record sets and their fields using mlcroissant's schema access

record_set_infos = []
record_sets = getattr(metadata, 'recordSet', [])
if not record_sets:
    # Try to find record sets via the dataset's schema property
    record_sets = getattr(metadata, 'record_sets', [])
    if not record_sets:
        # Fallback: Try dataset.record_sets (provided by mlcroissant)
        record_sets = dataset.record_sets

print("\nAvailable Record Sets:")
record_set_ids = []
for rs in record_sets:
    rs_id = getattr(rs, '@id', getattr(rs, 'id', str(rs)))
    rs_name = getattr(rs, 'name', None)
    print(f"  @id: {rs_id}   name: {rs_name}")
    if hasattr(rs, 'fields') and rs.fields is not None:
        print("    Fields:")
        for fld in rs.fields:
            f_id = getattr(fld, '@id', getattr(fld, 'id', str(fld)))
            f_name = getattr(fld, 'name', None)
            print(f"      @id: {f_id}   name: {f_name}")
    record_set_ids.append(rs_id)
if not record_set_ids:
    print("No record sets found with accessible fields.")

## 3. Data Extraction

Load data from each record set into Pandas DataFrames for analysis.

**Note:** Always reference record sets and fields by their `@id` fields for consistency. Update the code according to the available `@id`s listed above.

In [ ]:
# Collect data from each record set (by @id) into DataFrames
dataframes = {}

if not record_set_ids:
    print("No record sets available; please check dataset schema or contact the dataset curator.")
else:
    for rs_id in record_set_ids:
        try:
            print(f"Loading records from record set @id: {rs_id}")
            records = list(dataset.records(record_set=rs_id))
            if records:
                dataframes[rs_id] = pd.DataFrame(records)
                print(f"  Loaded {len(records)} records.")
            else:
                print("  No records loaded or record set is empty.")
        except Exception as e:
            print(f"  Error loading records for {rs_id}: {e}")

    if dataframes:
        # Choose the first record set with data for display
        first_rs_id = next((rid for rid in record_set_ids if rid in dataframes), None)
        if first_rs_id:
            print(f"\nColumns of '{first_rs_id}':")
            print(dataframes[first_rs_id].columns.tolist())
            print("\nFirst five records:")
            display(dataframes[first_rs_id].head())
    else:
        print("No dataframes populated. Schema may define no public record sets.")

## 4. Exploratory Data Analysis (EDA)

Apply example preprocessing and summarization steps:
  - Filter records based on a numeric field
  - Normalize the field
  - Optionally group by a categorical field

**Reference all columns by their `@id` fields.**

In [ ]:
# Identify a numeric and a categorical field (@id) from the first dataframe
import numpy as np

if not dataframes:
    print("No dataframes present. Run extraction cell first.")
else:
    # Try the first populated dataframe
    record_set_id = next(iter(dataframes))
    df = dataframes[record_set_id]

    # Heuristics: use the first float/int column as numeric_field, first object column as group_field
    numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
    group_cols = df.select_dtypes(include=["object"]).columns.tolist()

    if numeric_cols:
        numeric_field = numeric_cols[0]  # Use @id from schema if available
    else:
        numeric_field = None
    
    if group_cols:
        group_field = group_cols[0]  # Use @id
    else:
        group_field = None

    print(f"Numeric field selected: {numeric_field}")
    print(f"Grouping field selected: {group_field}")

    if numeric_field is not None:
        threshold = df[numeric_field].mean() if not np.isnan(df[numeric_field].mean()) else 0
        # Drop NaNs for demonstration
        valid_df = df.dropna(subset=[numeric_field])
        filtered_df = valid_df[valid_df[numeric_field] > threshold]
        print(f"\nFiltered records where {numeric_field} > {threshold:.2f}:")
        display(filtered_df.head())

        # Normalization
        colnorm = f"{numeric_field}_normalized"
        filtered_df[colnorm] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
        print(f"\nFirst rows with normalized {numeric_field}:")
        display(filtered_df[[numeric_field, colnorm]].head())

        # Group and summarize
        if group_field and group_field in filtered_df.columns:
            grouped_df = filtered_df.groupby(group_field)[numeric_field].mean()
            print(f"\nGrouped mean {numeric_field} by {group_field}:")
            display(grouped_df.head())
    else:
        print('No numeric field found to analyze.')

## 5. Visualization

Visualize distributions or field relationships for the selected record set.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if not dataframes or numeric_field is None:
    print("Skipping visualization; no suitable numeric field or dataframe loaded.")
else:
    plt.figure(figsize=(8,4))
    sns.histplot(df[numeric_field].dropna(), kde=True, bins=30)
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.ylabel('Frequency')
    plt.show()
    
    # If grouping field is present, boxplot
    if group_field and group_field in df.columns:
        plt.figure(figsize=(10,5))
        sns.boxplot(x=group_field, y=numeric_field, data=df)
        plt.title(f"{numeric_field} by {group_field}")
        plt.show()

## 6. Conclusion

In this notebook, we used the `mlcroissant` library to load and explore the FAIRˆ² dataset using its Croissant schema. We:
- Examined the dataset metadata and available record sets (referenced by `@id`).
- Loaded data from each record set and referenced fields by their `@id`.
- Applied basic EDA, filtering based on a numeric field, normalizing, and grouping by a key attribute.
- Visualized results with histograms and boxplots.
This approach can be extended to deeper analyses or other datasets described by Croissant schemas.